# 01 — Exploración de la Red Viaria de Madrid

Este notebook descarga la red viaria de Madrid desde OpenStreetMap, analiza su composición
y evalúa qué porcentaje de calles son transitables por camiones de bomberos según su anchura.

In [ ]:
import sys
sys.path.insert(0, '..')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pipeline.transform.build_graph import DEFAULT_WIDTH_BY_HIGHWAY, _get_width

## 1. Descarga de la red viaria

In [ ]:
# Descarga directa via osmnx (o carga desde caché si ya existe)
import os

CACHE_PATH = '../data/raw/osm/madrid_calles_raw.geojson'

if os.path.exists(CACHE_PATH):
    print('Cargando desde caché...')
    gdf = gpd.read_file(CACHE_PATH)
else:
    print('Descargando desde OSM (puede tardar ~5 min)...')
    G = ox.graph_from_place('Madrid, Community of Madrid, Spain', network_type='drive')
    _, gdf = ox.graph_to_gdfs(G)
    gdf = gdf.reset_index()

print(f'Aristas totales: {len(gdf):,}')
print(f'Columnas disponibles: {list(gdf.columns)}')

## 2. Composición por tipo de vía

In [ ]:
# Normalizar 'highway' (puede ser lista en OSM)
gdf['highway_norm'] = gdf['highway'].apply(
    lambda h: h[0] if isinstance(h, list) else h
)

counts = gdf['highway_norm'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
counts.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Distribución de tipos de vía en Madrid (OSM)', fontsize=13)
ax.set_xlabel('Número de segmentos')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

counts.head(15)

## 3. Análisis de anchuras

In [ ]:
# Estimar anchura para cada arista
gdf['width_m'] = gdf.apply(_get_width, axis=1)

print('Distribución de anchuras estimadas (metros):')
print(gdf['width_m'].describe())

fig, ax = plt.subplots(figsize=(9, 4))
gdf['width_m'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(3.5, color='red', linestyle='--', label='Mín. camión pesado (3.5m)')
ax.axvline(3.0, color='orange', linestyle='--', label='Mín. camión medio (3.0m)')
ax.set_title('Distribución de anchuras estimadas de la red viaria de Madrid')
ax.set_xlabel('Anchura (m)')
ax.set_ylabel('Número de segmentos')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Porcentaje de calles accesibles según tipo de camión
umbrales = {'Camión pesado (3.5m)': 3.5, 'Camión medio (3.0m)': 3.0, 'Vehículo ligero (2.5m)': 2.5}

total = len(gdf)
resumen = {}
for nombre, umbral in umbrales.items():
    accesibles = (gdf['width_m'] >= umbral).sum()
    resumen[nombre] = {'accesibles': accesibles, 'porcentaje': accesibles / total * 100}

df_resumen = pd.DataFrame(resumen).T
print(df_resumen.to_string())

## 4. Mapa de calles accesibles vs. bloqueadas

In [ ]:
# Proyectar a WGS84 para el mapa
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf_map = gdf.to_crs(epsg=4326)
else:
    gdf_map = gdf.copy()

ANCHO_REQ = 3.5
gdf_map['accesible'] = gdf_map['width_m'] >= ANCHO_REQ

fig, ax = plt.subplots(figsize=(12, 12))
gdf_map[~gdf_map['accesible']].plot(ax=ax, color='#ef4444', linewidth=0.4, alpha=0.6, label='Bloqueada')
gdf_map[gdf_map['accesible']].plot(ax=ax, color='#22c55e', linewidth=0.5, alpha=0.7, label='Accesible')
ax.set_title(f'Red viaria de Madrid — accesibilidad para camión ≥{ANCHO_REQ}m', fontsize=14)
ax.legend()
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 5. Construcción del grafo

In [ ]:
import networkx as nx
from pipeline.transform.build_graph import build_graph

# Reproyectar a UTM para el grafo
gdf_utm = gdf.to_crs(epsg=25830) if gdf.crs.to_epsg() != 25830 else gdf

G = build_graph(gdf_utm)

print(f'Nodos: {G.number_of_nodes():,}')
print(f'Aristas: {G.number_of_edges():,}')
print(f'Componentes fuertemente conexas: {nx.number_strongly_connected_components(G)}')
print(f'¿Es conexo?: {nx.is_weakly_connected(G)}')